# Glass-box baseline — EBM and rule-based models

Person 5 · CS3244 Group 1.

Trains and compares eight interpretable models on the SARC sarcasm corpus, picks
the best one, evaluates it on the held-out test set, and reads the fitted model
out term by term.

Write-ups: [`glassbox_report_en.md`](../../../docs/glassbox_report_en.md) ·
[`glassbox_report_zh.md`](../../../docs/glassbox_report_zh.md)

**Prerequisite:** `data/splits/` must exist. If it does not, run
`pipeline/notebooks/01`–`04` first.

**Runtime:** about 40 minutes with `QUICK = False`, about 4 minutes with
`QUICK = True`. Fitted models are cached, so re-running the notebook is fast.

**Reproducibility:** one seed (3244) drives sampling, every estimator and the
out-of-fold encoder. Feature selection, the SVD-free text representation and the
subreddit encoding are all fitted on the training split only. The test set is
read in exactly one place, section 6.

## 1. Setup

Imports, paths, and the switches that control this run. `QUICK` shrinks the data
and the search space for a smoke test; the reported numbers need `QUICK = False`.

In [ ]:
from __future__ import annotations

import hashlib, json, re, sys, time, warnings
from collections import Counter
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, average_precision_score,
                             confusion_matrix, f1_score, precision_recall_curve,
                             precision_score, recall_score, roc_auc_score, roc_curve)
from sklearn.model_selection import GroupKFold
from sklearn.tree import DecisionTreeClassifier, _tree

from interpret.glassbox import ExplainableBoostingClassifier
from imodels import (BoostedRulesClassifier, FIGSClassifier,
                     RuleFitClassifier, SkopeRulesClassifier)

warnings.filterwarnings("ignore")

QUICK = False
FORCE_RETRAIN = False
SEED = 3244

ROOT = Path.cwd()
while not (ROOT / "data" / "splits").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "data" / "splits").exists(), "run pipeline/notebooks/01-04 first"
sys.path.insert(0, str(ROOT / "pipeline"))
from load import load_split, load_lexicon, MODEL_SAFE_FEATURES

RESULTS = ROOT / "results" / "glassbox"
FIGURES = ROOT / "figures" / "glassbox"
MODELS  = ROOT / "models" / "glassbox" / "nb"
for d in (RESULTS, FIGURES, MODELS):
    d.mkdir(parents=True, exist_ok=True)

TRAIN_ROWS, VAL_ROWS = (20_000, 8_000) if QUICK else (200_000, 60_000)
N_UNIGRAMS, N_BIGRAMS, MIN_TERM_COUNT = 220, 80, 200
TARGET_PRECISION = 0.75
TRANSFER_TARGETS = ["figlang_reddit", "figlang_twitter",
                    "tweeteval_irony", "news_headlines"]

print(f"root      {ROOT}")
print(f"mode      {'QUICK' if QUICK else 'FULL'}  train={TRAIN_ROWS:,} val={VAL_ROWS:,}")

Record library versions. Results were produced with these; a different
`interpret` or `scikit-learn` may shift the last decimal.

In [ ]:
import importlib.metadata as md_
VERSIONS = {p: md_.version(p) for p in
            ["interpret", "imodels", "scikit-learn", "pandas", "numpy", "joblib"]}
VERSIONS["python"] = sys.version.split()[0]
pd.Series(VERSIONS).to_frame("version")

## 2. Data

`sarc_random` is the default split: grouped-random 70/15/15 with sibling comments
kept on the same side, so a parent comment never spans train and test. Training
is subsampled because EBM cost scales with rows; the test split is used whole.

In [ ]:
train, val, test = load_split("sarc_random")
rng = np.random.default_rng(SEED)

if TRAIN_ROWS < len(train):
    train = train.iloc[rng.choice(len(train), TRAIN_ROWS, replace=False)].reset_index(drop=True)
if VAL_ROWS < len(val):
    val = val.iloc[rng.choice(len(val), VAL_ROWS, replace=False)].reset_index(drop=True)

print(f"train {len(train):,}   val {len(val):,}   test {len(test):,}")
print(f"sarcastic rate — train {train.label.mean():.4f}  test {test.label.mean():.4f}")

## 3. Features

The whole point of this slot is that the model can be read, which only works if
its inputs can be read. Person 4's LightGBM used 300 truncated-SVD components —
fine for a tree ensemble, meaningless to a human. So the features are rebuilt
here as 340 named quantities in four blocks.

### 3.1 Lexical terms

The 300 most discriminative unigrams and bigrams, ranked by log-odds ratio with
an informative Dirichlet prior (Monroe et al. 2008). Frequency alone returns
stopwords; a raw ratio returns typos. Selection uses the training split only.

In [ ]:
RE_TOKEN = re.compile(r"[a-z][a-z']+")
tokens   = lambda t: RE_TOKEN.findall(t.lower())
bigrams  = lambda tk: [f"{a}_{b}" for a, b in zip(tk, tk[1:])]

def log_odds_terms(pos, neg, n_uni, n_bi, min_count):
    cu_p, cu_n, cb_p, cb_n = Counter(), Counter(), Counter(), Counter()
    for t in pos:
        tk = tokens(t); cu_p.update(tk); cb_p.update(bigrams(tk))
    for t in neg:
        tk = tokens(t); cu_n.update(tk); cb_n.update(bigrams(tk))

    def rank(cp, cn, k):
        prior = cp + cn
        vocab = [w for w in prior if prior[w] >= min_count]
        na, nb, n0 = sum(cp.values()), sum(cn.values()), sum(prior.values())
        a0 = n0 * 0.01
        out = []
        for w in vocab:
            ya, yb, aw = cp[w], cn[w], prior[w] / n0 * a0
            oa = np.log((ya + aw) / (na + a0 - ya - aw))
            ob = np.log((yb + aw) / (nb + a0 - yb - aw))
            out.append((w, float((oa - ob) / np.sqrt(1/(ya+aw) + 1/(yb+aw)))))
        return sorted(out, key=lambda x: -abs(x[1]))[:k]

    return rank(cu_p, cu_n, n_uni), rank(cb_p, cb_n, n_bi)

uni, bi = log_odds_terms(train.loc[train.label == 1, "comment_clean"].tolist(),
                         train.loc[train.label == 0, "comment_clean"].tolist(),
                         N_UNIGRAMS, N_BIGRAMS, MIN_TERM_COUNT)
UNIGRAMS  = [w for w, _ in uni]
BIGRAMS   = [w for w, _ in bi]

print("sarcastic:", ", ".join(w for w, z in uni[:14] if z > 0))
print("sincere:  ", ", ".join(w for w, z in uni[:30] if z < 0)[:90])

### 3.2 Lexicon features

VADER valence and NRC emotion densities for the reply and the parent, plus the
sentiment-incongruity terms — a warm reply under a hostile parent, which is the
most cited engineered cue in the sarcasm literature.

In [ ]:
EMOTIONS = ["anger", "anticipation", "disgust", "fear", "joy",
            "sadness", "surprise", "trust", "positive", "negative"]
LEXICON_NAMES = ([f"nrc_{e}" for e in EMOTIONS] +
                 ["vader_mean", "vader_pos_share", "vader_neg_share",
                  "parent_vader_mean", "incongruity_pos_reply_neg_parent",
                  "incongruity_signed"])

VADER = load_lexicon("vader")
_nrc  = load_lexicon("nrc")
NRC   = {e: _nrc.get(e, set()) for e in EMOTIONS}

def _vader(tk):
    vals = [VADER[w] for w in tk if w in VADER]
    if not vals:
        return 0.0, 0.0, 0.0
    a = np.asarray(vals, dtype=np.float32)
    return float(a.mean()), float((a > 0).mean()), float((a < 0).mean())

def lexicon_block(text, parent):
    out = np.zeros((len(text), len(LEXICON_NAMES)), dtype=np.float32)
    k = len(EMOTIONS)
    for r, (t, p) in enumerate(zip(text.fillna(""), parent.fillna(""))):
        tk = tokens(t); s = set(tk); n = max(1, len(tk))
        for j, e in enumerate(EMOTIONS):
            out[r, j] = len(s & NRC[e]) / n
        m, pos, neg = _vader(tk)
        pm, _, _ = _vader(tokens(p))
        out[r, k:k+6] = [m, pos, neg, pm,
                         1.0 if (m > 0.05 and pm < -0.05) else 0.0, m * pm]
    return out

print(f"{len(LEXICON_NAMES)} lexicon features | VADER {len(VADER):,} tokens")

### 3.3 Subreddit encoding

Out-of-fold target encoding with folds grouped by `parent_key`. Naive target
encoding leaks each row's own label; the printed correlations show how much that
would have been worth. Unseen subreddits fall back to the training prior.

In [ ]:
class SubredditEncoder:
    def __init__(self, smoothing=50.0, n_splits=5):
        self.smoothing, self.n_splits = smoothing, n_splits
        self.prior_, self.mapping_, self.freq_ = 0.5, {}, {}

    def _smooth(self, agg):
        return (agg["mean"] * agg["count"] + self.prior_ * self.smoothing) \
               / (agg["count"] + self.smoothing)

    def fit_transform_train(self, sub, y, groups):
        self.prior_ = float(y.mean())
        oof = np.full(len(sub), self.prior_, dtype=np.float32)
        for tr, va in GroupKFold(n_splits=self.n_splits).split(sub, y, groups=groups):
            agg = (pd.DataFrame({"s": sub.iloc[tr].to_numpy(), "y": y.iloc[tr].to_numpy()})
                     .groupby("s")["y"].agg(["count", "mean"]))
            oof[va] = sub.iloc[va].map(self._smooth(agg)).fillna(self.prior_).to_numpy("float32")
        full = (pd.DataFrame({"s": sub.to_numpy(), "y": y.to_numpy()})
                  .groupby("s")["y"].agg(["count", "mean"]))
        self.mapping_, self.freq_ = self._smooth(full).to_dict(), full["count"].to_dict()
        return oof

    def transform(self, sub):
        return sub.map(self.mapping_).fillna(self.prior_).to_numpy("float32")

    def frequency(self, sub):
        return np.log1p(sub.map(self.freq_).fillna(0).to_numpy("float32"))

ENC = SubredditEncoder()
OOF_TE = ENC.fit_transform_train(train.subreddit, train.label, train.parent_key)

print(f"prior {ENC.prior_:.4f} | {len(ENC.mapping_):,} communities")
print(f"corr with label — out-of-fold {np.corrcoef(OOF_TE, train.label)[0,1]:.4f}"
      f"  vs in-fold {np.corrcoef(ENC.transform(train.subreddit), train.label)[0,1]:.4f}")

### 3.4 Assemble

Stack the four blocks into one matrix. Every column has a name that reads as a
statement about the comment.

In [ ]:
FEATURE_NAMES = (list(MODEL_SAFE_FEATURES)
                 + [f"has:{w}" for w in UNIGRAMS]
                 + [f"has:{w.replace('_', ' ')}" for w in BIGRAMS]
                 + LEXICON_NAMES + ["subreddit_rate", "subreddit_logfreq"])
n_surf, n_lex = len(MODEL_SAFE_FEATURES), len(UNIGRAMS) + len(BIGRAMS)
BLOCKS = {"SURFACE": (0, n_surf), "LEXICAL": (n_surf, n_surf + n_lex),
          "LEXICON": (n_surf + n_lex, n_surf + n_lex + len(LEXICON_NAMES)),
          "SUBREDDIT": (len(FEATURE_NAMES) - 2, len(FEATURE_NAMES))}

UNI_IDX = {w: i for i, w in enumerate(UNIGRAMS)}
BI_IDX  = {w: i + len(UNIGRAMS) for i, w in enumerate(BIGRAMS)}

def build_X(df, sub_te=None):
    tcol = "comment_clean" if "comment_clean" in df else "text_clean"
    pcol = next((c for c in ("parent_clean", "context_clean") if c in df), None)
    text = df[tcol]
    parent = df[pcol] if pcol else pd.Series([""] * len(df), index=df.index)

    surface = np.zeros((len(df), n_surf), dtype=np.float32)
    for j, c in enumerate(MODEL_SAFE_FEATURES):
        if c in df:
            surface[:, j] = pd.to_numeric(df[c], errors="coerce").astype("float32").fillna(0).to_numpy()

    lexical = np.zeros((len(df), n_lex), dtype=np.float32)
    for r, t in enumerate(text.fillna("")):
        tk = tokens(t)
        for w in set(tk):
            if (j := UNI_IDX.get(w)) is not None: lexical[r, j] = 1.0
        for w in set(bigrams(tk)):
            if (j := BI_IDX.get(w)) is not None: lexical[r, j] = 1.0

    if "subreddit" in df:
        te = sub_te if sub_te is not None else ENC.transform(df.subreddit)
        freq = ENC.frequency(df.subreddit)
    else:
        te, freq = np.full(len(df), ENC.prior_, "float32"), np.zeros(len(df), "float32")

    return np.hstack([surface, lexical, lexicon_block(text, parent),
                      np.column_stack([te, freq]).astype(np.float32)])

t0 = time.time()
Xtr, ytr = build_X(train, OOF_TE), train.label.to_numpy(np.int8)
Xva, yva = build_X(val),           val.label.to_numpy(np.int8)
Xte, yte = build_X(test),          test.label.to_numpy(np.int8)

FEATURE_FINGERPRINT = hashlib.md5(
    f"{QUICK}|{TRAIN_ROWS}|{VAL_ROWS}|{'|'.join(FEATURE_NAMES)}".encode()).hexdigest()[:10]

print(f"{len(FEATURE_NAMES)} features in {len(BLOCKS)} blocks  ({time.time()-t0:.0f}s)")
print(f"train {Xtr.shape}   val {Xva.shape}   test {Xte.shape}")
print(f"fingerprint {FEATURE_FINGERPRINT} — model cache is keyed on this, so a "
      f"QUICK run can never be reused by a full run")

## 4. Mine readable rules

Short conjunctions extracted from a shallow boosted ensemble, kept only if they
fire on between 1% and 90% of rows. These feed one of the combination candidates
and are reported in section 8 as a description of the data.

In [ ]:
def mine_rules(X, y, n_estimators=30, max_depth=3, max_rules=80):
    gb = GradientBoostingClassifier(n_estimators=n_estimators, max_depth=max_depth,
                                    learning_rate=0.1, subsample=0.7,
                                    random_state=SEED).fit(X, y)
    raw = []
    for est in gb.estimators_.ravel():
        tr = est.tree_
        def walk(node, conds):
            if tr.feature[node] == _tree.TREE_UNDEFINED:
                if conds: raw.append(tuple(conds))
                return
            f, t = int(tr.feature[node]), float(tr.threshold[node])
            walk(tr.children_left[node],  conds + [(f, "<=", t)])
            walk(tr.children_right[node], conds + [(f, ">",  t)])
        walk(0, [])

    seen, kept = set(), []
    for r in raw:
        key = tuple(sorted((f, o, round(t, 6)) for f, o, t in r))
        if key in seen: continue
        seen.add(key)
        m = apply_rule(X, r)
        if 0.01 <= m.mean() <= 0.90:
            kept.append((r, abs(y[m].mean() - y.mean())))
    kept.sort(key=lambda x: -x[1])
    return [r for r, _ in kept[:max_rules]]

def apply_rule(X, rule):
    m = np.ones(len(X), dtype=bool)
    for f, op, t in rule:
        m &= (X[:, f] <= t) if op == "<=" else (X[:, f] > t)
    return m

def render_rule(rule):
    parts = []
    for f, op, t in rule:
        n = FEATURE_NAMES[f]
        if n.startswith("has:"):
            parts.append(f"contains '{n[4:]}'" if op == ">" else f"no '{n[4:]}'")
        else:
            parts.append(f"{n} {op} {t:.3g}")
    return " AND ".join(parts)

def rule_matrix(X, rs):
    if not rs:
        return np.zeros((len(X), 0), dtype=np.float32)
    return np.column_stack([apply_rule(X, r) for r in rs]).astype(np.float32)

t0 = time.time()
RULES = mine_rules(Xtr, ytr)
print(f"{len(RULES)} rules ({time.time()-t0:.0f}s)")
for r in RULES[:4]:
    print(" ·", render_rule(r))

## 5. Train the candidates

Eight glass-box models. Two of them are combinations: `ebm_plus_rules` adds the
mined conjunctions as extra EBM inputs, and `ebm_rulefit_avg` averages the two
strongest components' probabilities. Both stay glass-box because every part can
still be printed in full.

Selection is on validation ROC-AUC. AUC is threshold-free, so it ranks models on
ranking quality and leaves the precision/recall trade-off to section 6.

Fitted models are cached to `models/glassbox/nb/`; set `FORCE_RETRAIN = True` to
refit.

In [ ]:
EBM_FIXED = dict(random_state=SEED, n_jobs=-1, outer_bags=4, inner_bags=0,
                 early_stopping_rounds=50, max_rounds=5000)

GRIDS = {
    "decision_tree_d4": [{"max_depth": d} for d in (3, 4, 5, 6)],
    "figs":             [{"max_rules": 12}, {"max_rules": 20}],
    "boosted_rules":    [{"n_estimators": n} for n in (20, 40, 80)],
    "skope_rules":      [{"n_estimators": 20, "max_depth": 3},
                         {"n_estimators": 40, "max_depth": 4}],
    "rulefit":          [{"max_rules": 60, "tree_size": 4}],
    "ebm":              [{"interactions": 0,  "max_bins": 256, "learning_rate": 0.02},
                         {"interactions": 10, "max_bins": 256, "learning_rate": 0.02},
                         {"interactions": 20, "max_bins": 256, "learning_rate": 0.02},
                         {"interactions": 20, "max_bins": 512, "learning_rate": 0.01}],
    "ebm_plus_rules":   [{"interactions": 10, "max_bins": 256, "learning_rate": 0.02}],
}
if QUICK:
    GRIDS = {k: v[:1] for k, v in GRIDS.items()}

def make(name, params):
    if name == "decision_tree_d4":
        return DecisionTreeClassifier(random_state=SEED, min_samples_leaf=200, **params)
    if name == "figs":          return FIGSClassifier(random_state=SEED, **params)
    if name == "boosted_rules": return BoostedRulesClassifier(random_state=SEED, **params)
    if name == "skope_rules":   return SkopeRulesClassifier(random_state=SEED, **params)
    if name == "rulefit":       return RuleFitClassifier(random_state=SEED, **params)
    return ExplainableBoostingClassifier(**EBM_FIXED, **params)

def proba_of(m, X):
    p = np.asarray(m.predict_proba(X))
    return p[:, 1] if p.ndim == 2 and p.shape[1] == 2 else p.ravel()

def metrics(y, p, thr=0.5):
    q = (p >= thr).astype(int)
    return {"roc_auc": round(float(roc_auc_score(y, p)), 5),
            "accuracy": round(float(accuracy_score(y, q)), 4),
            "precision": round(float(precision_score(y, q, zero_division=0)), 4),
            "recall": round(float(recall_score(y, q, zero_division=0)), 4),
            "f1": round(float(f1_score(y, q, zero_division=0)), 4)}

def complexity(name, m, params):
    try:
        if name == "decision_tree_d4":
            return f"{m.get_n_leaves()} leaves, depth {m.get_depth()}"
        if name.startswith("ebm"):
            k = sum(1 for t in m.term_features_ if len(t) > 1)
            return f"{len(m.term_features_)} terms ({k} interactions)"
        if name == "rulefit":
            return f"{len(m.rules_)} rules kept"
    except Exception:
        pass
    return str(params)

Run the sweep. Roughly 35 minutes in full mode, dominated by the four EBM
configurations and RuleFit.

In [ ]:
Rtr, Rva = rule_matrix(Xtr, RULES), rule_matrix(Xva, RULES)
trials, best = [], {}

for name, grid in GRIDS.items():
    for params in grid:
        tag = f"{name}|{params}|{FEATURE_FINGERPRINT}"
        cache = MODELS / f"{name}_{hashlib.md5(tag.encode()).hexdigest()[:8]}.joblib"
        t0 = time.time()
        if cache.exists() and not FORCE_RETRAIN:
            model, secs = joblib.load(cache), 0.0
        else:
            try:
                model = make(name, params).fit(
                    np.hstack([Xtr, Rtr]) if name == "ebm_plus_rules" else Xtr, ytr)
            except Exception as e:
                print(f"  {name:<18} {str(params):<48} FAILED {type(e).__name__}")
                continue
            joblib.dump(model, cache, compress=3)
            secs = time.time() - t0
        p = proba_of(model, np.hstack([Xva, Rva]) if name == "ebm_plus_rules" else Xva)
        m = metrics(yva, p)
        cx = complexity(name, model, params)
        trials.append({"model": name, "params": str(params), "seconds": round(secs, 1),
                       "complexity": cx, **m})
        if name not in best or m["roc_auc"] > best[name]["score"]["roc_auc"]:
            best[name] = {"model": model, "params": params, "score": m,
                          "proba": p, "complexity": cx}
        print(f"  {name:<18} {str(params):<48} AUC={m['roc_auc']:.5f} {secs:>6.0f}s")

if "ebm" in best and "rulefit" in best:
    p_avg = 0.5 * best["ebm"]["proba"] + 0.5 * best["rulefit"]["proba"]
    m_avg = metrics(yva, p_avg)
    trials.append({"model": "ebm_rulefit_avg", "params": "0.5/0.5", "seconds": 0.0,
                   "complexity": "avg of ebm and rulefit", **m_avg})
    best["ebm_rulefit_avg"] = {"model": None, "params": {"w": 0.5}, "score": m_avg,
                               "proba": p_avg, "complexity": "avg of ebm and rulefit"}
    print(f"  {'ebm_rulefit_avg':<18} {'0.5/0.5':<48} AUC={m_avg['roc_auc']:.5f}")

Leaderboard. The top two are separated by less than a thousandth of AUC, which
matters for the recommendation in section 6.

In [ ]:
board = pd.DataFrame([{"model": k, **v["score"], "complexity": v["complexity"]}
                      for k, v in best.items()]).sort_values("roc_auc", ascending=False)
WINNER = board.iloc[0]["model"]
display(board.reset_index(drop=True))
print(f"winner: {WINNER}   margin over runner-up: "
      f"{board.iloc[0]['roc_auc'] - board.iloc[1]['roc_auc']:+.5f} AUC")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.2))
d = board.sort_values("roc_auc")
ax.barh(np.arange(len(d)), d.roc_auc,
        color=["#eb6834" if i == len(d)-1 else "#2a78d6" for i in range(len(d))])
for i, v in enumerate(d.roc_auc):
    ax.annotate(f"{v:.4f}", (v, i), xytext=(5, 0), textcoords="offset points",
                va="center", fontsize=9)
ax.set_yticks(np.arange(len(d)), d.model)
ax.set_xlim(0.5, 0.80); ax.axvline(0.5, ls="--", lw=1, c="#8a8982")
ax.set_xlabel("Validation ROC-AUC"); ax.set_title("Glass-box candidates")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

## 6. Test-set evaluation

The threshold comes from validation: the lowest point reaching precision ≥ 0.75.
The proposal treats a false "sarcastic" call as the expensive error, so the
operating point targets precision rather than F1 — maximising F1 would weight
both errors equally and actually lowers precision below the 0.5 default.

Both finalists go to test, because a 0.0004 validation gap is not enough to
justify the more complex one.

**This is the only cell that reads the test set.**

In [ ]:
Rte = rule_matrix(Xte, RULES)

def predict(name, X, R):
    if name == "ebm_rulefit_avg":
        return 0.5 * predict("ebm", X, R) + 0.5 * predict("rulefit", X, R)
    Xin = np.hstack([X, R]) if name == "ebm_plus_rules" else X
    return proba_of(best[name]["model"], Xin)

def pick_threshold(y, p, target):
    prec, rec, thr = precision_recall_curve(y, p)
    prec, rec = prec[:-1], rec[:-1]
    ok = np.where(prec >= target)[0]
    i = int(ok[np.argmax(rec[ok])]) if len(ok) else int(np.argmax(2*prec*rec/(prec+rec+1e-12)))
    return float(thr[i]), float(prec[i]), float(rec[i])

finalists = [WINNER] + [m for m in board.model[1:2] if m != WINNER]
rows = []
for name in finalists:
    thr, vp, vr = pick_threshold(yva, predict(name, Xva, Rva), TARGET_PRECISION)
    p_te = predict(name, Xte, Rte)
    for label, t in (("@threshold", thr), ("@0.50", 0.5)):
        rows.append({"model": name, "point": label, "thr": round(thr if label != "@0.50" else 0.5, 4),
                     **metrics(yte, p_te, t),
                     "pr_auc": round(float(average_precision_score(yte, p_te)), 4)})
    if name == WINNER:
        THR, P_TE = thr, p_te

rows.append({"model": "LightGBM (Person 4, black box)", "point": "@0.50", "thr": 0.5,
             "roc_auc": 0.7803, "accuracy": 0.7059, "precision": 0.7285,
             "recall": 0.6760, "f1": 0.7013, "pr_auc": None})
rows.append({"model": "majority baseline", "point": "—", "thr": None,
             "roc_auc": 0.5, "accuracy": round(float(max(yte.mean(), 1-yte.mean())), 4),
             "precision": None, "recall": None, "f1": None, "pr_auc": None})
display(pd.DataFrame(rows))

tn, fp, fn, tp = confusion_matrix(yte, (P_TE >= 0.5).astype(int)).ravel()
print(f"confusion @0.50 (n={len(yte):,}): TN {tn:,}  FP {fp:,}  FN {fn:,}  TP {tp:,}")

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.2))
fpr, tpr, _ = roc_curve(yte, P_TE)
a1.plot(fpr, tpr, c="#eb6834", lw=2, label=f"AUC {roc_auc_score(yte, P_TE):.3f}")
a1.plot([0, 1], [0, 1], ls="--", lw=1, c="#8a8982")
a1.set_xlabel("False positive rate"); a1.set_ylabel("True positive rate")
a1.set_title("ROC — test"); a1.legend(loc="lower right")

prec, rec, _ = precision_recall_curve(yte, P_TE)
a2.plot(rec, prec, c="#eb6834", lw=2,
        label=f"AP {average_precision_score(yte, P_TE):.3f}")
a2.axhline(yte.mean(), ls="--", lw=1, c="#8a8982")
a2.set_xlabel("Recall"); a2.set_ylabel("Precision")
a2.set_title("Precision-recall — test"); a2.legend(loc="lower left")
for a in (a1, a2): a.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

## 7. Transfer off Reddit

Same model, same threshold, four independent corpora. Read AUC rather than F1
here: the threshold was calibrated on SARC, and off-domain the probability
distribution shifts down, so F1 collapses for reasons of calibration rather than
capability.

In [ ]:
xfer = [{"target": "sarc_random (in-domain)", **metrics(yte, P_TE, THR),
         "majority": round(float(max(yte.mean(), 1-yte.mean())), 4)}]
for name in TRANSFER_TARGETS:
    d = load_split(name, "test")
    X = build_X(d); y = d.label.to_numpy(np.int8)
    xfer.append({"target": name, **metrics(y, predict(WINNER, X, rule_matrix(X, RULES)), THR),
                 "majority": round(float(max(y.mean(), 1-y.mean())), 4)})
xfer = pd.DataFrame(xfer)
display(xfer)

fig, ax = plt.subplots(figsize=(8.6, 4))
ax.bar(xfer.target, xfer.roc_auc,
       color=["#eb6834" if v >= 0.6 else "#2a78d6" for v in xfer.roc_auc])
for i, v in enumerate(xfer.roc_auc):
    ax.annotate(f"{v:.3f}", (i, v), ha="center", xytext=(0, 3),
                textcoords="offset points", fontsize=9)
ax.axhline(0.5, ls="--", lw=1.2, c="#0b0b0b")
ax.set_ylim(0.4, 0.85); ax.set_ylabel("ROC-AUC")
ax.set_title("Transfer — below the line is worse than chance")
ax.tick_params(axis="x", rotation=20)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

## 8. Read the model

This is what the other base-model slots cannot do. LightGBM needs permutation
importance or SHAP, which *estimate* what a model relies on. An EBM is a sum of
one curve per feature, so the curves are the model exactly.

### 8.1 Term importance

In [ ]:
ebm = best["ebm"]["model"]
data = ebm.explain_global().data()

def block_of(i):
    return next((b for b, (lo, hi) in BLOCKS.items() if lo <= i < hi), "OTHER")

imp = []
for i, t in enumerate(ebm.term_features_):
    imp.append({"term": FEATURE_NAMES[t[0]] if len(t) == 1
                        else " x ".join(FEATURE_NAMES[j] for j in t),
                "block": block_of(t[0]) if len(t) == 1 else "INTERACTION",
                "feature_idx": t[0] if len(t) == 1 else -1,
                "importance": float(data["scores"][i])})
imp = pd.DataFrame(imp).sort_values("importance", ascending=False)

share = imp.groupby("block").importance.sum()
display(imp.head(20).reset_index(drop=True))
display((share / share.sum()).sort_values(ascending=False).to_frame("share of model"))

### 8.2 Shape functions

The learned curve for each feature. Two things to look for: the exclamation-mark
curve peaking at two or three and falling back past four, and the length curves
carrying more weight than any lexical term despite the EDA finding both classes
have the same median length. Both statements are true — the EDA measured the
marginal distribution, this measures the conditional effect.

In [ ]:
top = [r.feature_idx for r in imp.itertuples()
       if r.block in ("SURFACE", "LEXICON", "SUBREDDIT") and r.feature_idx >= 0][:9]

fig, axes = plt.subplots(3, 3, figsize=(13, 9))
for ax, idx in zip(axes.ravel(), top):
    ti = ebm.term_features_.index((idx,))
    sc = np.asarray(ebm.term_scores_[ti])[1:-1]
    edges = np.asarray(ebm.bins_[idx][0], dtype=float)
    ctr = np.concatenate([[edges[0]], (edges[:-1]+edges[1:])/2, [edges[-1]]])[:len(sc)]
    ax.step(ctr, sc[:len(ctr)], where="mid", c="#eb6834", lw=2)
    ax.axhline(0, ls="--", lw=1, c="#8a8982")
    ax.set_title(FEATURE_NAMES[idx].replace("has:", ""), fontsize=10)
    ax.set_ylabel("log-odds", fontsize=8)
    ax.spines[["top", "right"]].set_visible(False)
fig.suptitle("Shape functions — above zero pushes toward 'sarcastic'",
             x=0.01, ha="left", fontsize=12)
plt.tight_layout(rect=(0, 0, 1, 0.97)); plt.show()

### 8.3 Rules

Coverage and sarcasm rate measured on validation. The rule set as a whole loses
to the EBM by six AUC points, but individual rules are sharp — useful as findings
rather than as a classifier.

In [ ]:
rt = [{"rule": render_rule(r), "coverage": round(float(m.mean()), 4),
       "sarcasm_rate": round(float(yva[m].mean()), 4), "n": int(m.sum())}
      for r in RULES if (m := apply_rule(Xva, r)).any()]
rt = pd.DataFrame(rt).sort_values("sarcasm_rate", ascending=False)

print("strongest sarcasm rules")
display(rt.head(6).reset_index(drop=True))
print("strongest sincerity rules")
display(rt.tail(4).reset_index(drop=True))

## 9. Save

Writes the tables and the run manifest so the report can cite exact numbers, and
so a later run can be diffed against this one.

In [ ]:
board.to_csv(RESULTS / "nb_leaderboard.csv", index=False)
pd.DataFrame(trials).to_csv(RESULTS / "nb_candidate_trials.csv", index=False)
pd.DataFrame(rows).to_csv(RESULTS / "nb_test_results.csv", index=False)
xfer.to_csv(RESULTS / "nb_transfer.csv", index=False)
imp.to_csv(RESULTS / "nb_term_importance.csv", index=False)
rt.to_csv(RESULTS / "nb_rules.csv", index=False)

(RESULTS / "nb_run.json").write_text(json.dumps({
    "generated_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "quick_mode": QUICK, "seed": SEED, "versions": VERSIONS,
    "train_rows": int(len(ytr)), "val_rows": int(len(yva)), "test_rows": int(len(yte)),
    "n_features": len(FEATURE_NAMES), "n_rules": len(RULES),
    "winner": WINNER, "threshold": round(THR, 4),
    "block_share": {k: round(float(v), 4) for k, v in (share / share.sum()).items()},
}, indent=2), encoding="utf-8")

print(f"saved 6 tables + manifest to {RESULTS}")

## Takeaways

- **EBM wins, and the runner-up is EBM without the rule indicators.** The gap is
  0.0004 on validation and 0.0005 on test — noise. Ship plain EBM: fewer terms,
  faster, and slightly better accuracy and F1.
- **Full interpretability costs about 1.4 accuracy points** against the black-box
  LightGBM (0.6915 vs 0.7059), with every input a phrase a human can read.
- **Rule learners lose by roughly six AUC points.** Sarcasm signal here is
  diffuse; no small rule set captures it. Individual rules still reach 92%
  precision on the slices they cover.
- **Averaging EBM with RuleFit makes things worse.** Feeding a weak model's
  features to a strong one is safe; averaging their predictions is not.
- **The exclamation-mark curve is non-monotonic** — one to three marks push
  toward sarcasm, four or more fall back. The EDA's single 2.81× lift averaged
  that away, and only a shaped model can show it.
- Attribution describes the *model*, not sarcasm. `subreddit_rate` carries weight
  because the `/s` convention varies by community, not because community causes
  sarcasm.